# 01 – Evolvente und Bezugsprofil

**Ziel.** Dieses Notebook dokumentiert die Grundgrößen des Einzelrades (Teilkreis, Grundkreis, Stirneingriffswinkel, Grundschrägungswinkel), die Evolventenfunktion mit ihrer Umkehrung, Zahndicke und Lückenweite sowie die Bezugsprofile von Verzahnung und Werkzeug. Es sichert die Module `gearcore.involute`, `gearcore.rack` und `gearcore.parity` ab.

**Regeln.** Das Notebook ist ausführbare Dokumentation (ADR-104): Es ruft `gearcore` auf und definiert selbst keine Formeln. Die Rechenvorschriften stehen genau einmal im Paket (`gearcore.involute`, `gearcore.rack`); dieselben Funktionen nutzen später die Paarungsgeometrie, die Schnittstelle und die Oberfläche. Programmnamen, Formelzeichen und Benennungen stehen genau einmal in der Größen-Registry (`data/quantities.yaml`, Projektregel 3b). Jede Aussage trägt Norm, Ausgabe, Abschnitt, Gleichung und Seite. Normtext wird nicht wörtlich wiedergegeben; die Formeln sind eigene Transkriptionen in der Schreibweise der aktuell gültigen Normen (Projektregel 3a). Gerechnet wird durchgängig mit doppelter Genauigkeit (64 Bit), gerundet wird nur in der Anzeige (Projektregel 8a, ADR-106).

**Quellen.**

| Schlüssel in `sources.yaml` | Norm | Inhalt in diesem Notebook |
|---|---|---|
| `ISO21771:2014` | DIN ISO 21771:2014-08 | §4.2, §4.3, §4.7: Bestimmungsgrößen, Evolvente, Zahndicke; §7.1: Werkzeug-Bezugsprofil |
| `DIN867:1986` | DIN 867:1986-02 | Bezugsprofil der Verzahnung |
| `ISO53:1998` | ISO 53:1998 | Bezugsprofil-Typen A bis D |
| `DIN3972:1952` | DIN 3972:1952-02 | Bezugsprofile I bis IV der Verzahnwerkzeuge |
| `ISO54:1996` | ISO 54:1996 | Modulreihe |
| `ISOTR6336-30:2022` | ISO/TR 6336-30:2022 | Rechenbeispiel 1 (Anhang A) als Referenz |

In [ ]:
import inspect
import math

import matplotlib.pyplot as plt
import pandas as pd

import gearcore
from gearcore import involute as iv
from gearcore import rack
from gearcore.data import load_worked_example, printed_tolerance, stplus_case_dirs
from gearcore.parity import ParityVerdict, compare_basic_geometry
from gearcore.quantities import latex, quantities
from gearcore.trace import equations_of

pd.set_option("display.precision", 6)
print("gearcore", gearcore.__version__)

## 1 Bestimmungsgrößen des Einzelrades

**Normbezug.** DIN ISO 21771:2014-08: §4.2.4 Gl. (1), S. 23; §4.2.7 Gl. (2), S. 24; §4.3.3 Gl. (6), S. 28; §4.3.6 Gl. (14), S. 30; §4.3.10 Gl. (19), S. 31.

**Formeln (eigene Transkription).**

$$d = \frac{z\,m_{\mathrm{n}}}{\cos\beta} \quad (1) \qquad m_{\mathrm{t}} = \frac{m_{\mathrm{n}}}{\cos\beta} \quad (2) \qquad \tan\alpha_{\mathrm{t}} = \frac{\tan\alpha_{\mathrm{n}}}{\cos\beta} \quad (14)$$

$$\sin\beta_{\mathrm{b}} = \sin\beta\,\cos\alpha_{\mathrm{n}} \quad (6) \qquad d_{\mathrm{b}} = d\,\cos\alpha_{\mathrm{t}} \quad (19)$$

Der Stirnschnitt ist die Arbeitsebene: Alle Größen gelten für Gerad- ($\beta = 0$) und Schrägverzahnung. Behandelt wird die Außenverzahnung; die Innenverzahnung ist ein vorbereiteter Erweiterungspunkt.

**Referenz.** ISO/TR 6336-30:2022, Anhang A, Beispiel 1 (schrägverzahnt, $z$ = 17/103, $m_{\mathrm{n}}$ = 8 mm, $\beta$ = 15,8°). Die Referenzwerte liegen als Fixture im Paket; die Toleranz ist eine halbe Einheit der letzten gedruckten Stelle.

In [ ]:
example = load_worked_example("iso_tr_6336_30_2022_annex_a_example_1")
inputs, expected = example["inputs"], example["expected"]
print(example["source"], "–", example["location"])

# Eingaben wie in Table A.1 gedruckt (Benennung, Formelzeichen, gedruckter Wert)
given = pd.DataFrame(
    [
        {
            "Benennung der Norm": entry["description"],
            "Zeichen": entry["symbol"] or "–",
            "gedruckt": entry["printed"]
            if isinstance(entry["printed"], str)
            else " | ".join(entry["printed"].values()),
        }
        for entry in inputs.values()
    ]
)
print(given.to_string(index=False))

gears = {
    role: iv.compute_basic_gear_geometry(
        number_of_teeth=inputs["number_of_teeth"][role],
        normal_module_mm=inputs["normal_module_mm"]["value"],
        normal_pressure_angle_deg=inputs["normal_pressure_angle_deg"]["value"],
        helix_angle_deg=inputs["helix_angle_deg"]["value"],
        profile_shift_coefficient=inputs["profile_shift_coefficient"][role],
    )
    for role in ("pinion", "wheel")
}

rows = []
for field in (
    "transverse_module_mm",
    "transverse_pressure_angle_deg",
    "base_helix_angle_deg",
    "reference_diameter_mm",
    "base_diameter_mm",
):
    entry = expected[field]
    tolerance = printed_tolerance(entry["decimals"])
    for role, gear in gears.items():
        reference = entry[role] if role in entry else entry["value"]
        value = getattr(gear, field)
        assert abs(value - reference) <= tolerance + 1e-9, (field, role, value, reference)
        rows.append(
            {
                "Benennung der Norm": entry["description"],
                "Zeichen": entry["symbol"],
                "Rad": role,
                "gearcore": value,
                "Norm (gedruckt)": entry["printed"][role] if role in entry else entry["printed"],
                "Abweichung": value - reference,
                "Toleranz": tolerance,
            }
        )
pd.DataFrame(rows)

## 2 Evolventenfunktion und ihre Umkehrung

**Normbezug.** DIN ISO 21771:2014-08: §4.3.5 Gl. (12), S. 29; §4.3.7 Gl. (16), S. 30; §4.3.8 Gl. (17), S. 30; §4.3.9 Gl. (18), S. 31.

**Formeln (eigene Transkription).**

$$\operatorname{inv}\alpha = \tan\alpha - \alpha \quad (18) \qquad \cos\alpha_{\mathrm{yt}} = \frac{d_{\mathrm{b}}}{d_{\mathrm{y}}} \quad (12) \qquad \xi_{\mathrm{y}} = \tan\alpha_{\mathrm{yt}} \quad (16) \qquad \rho_{\mathrm{y}} = \tfrac{1}{2}\sqrt{d_{\mathrm{y}}^2 - d_{\mathrm{b}}^2} \quad (17)$$

**Umkehrung.** Die Norm gibt keine Umkehrfunktion an. `gearcore` enthält genau eine numerische Umkehrung (`inv_inverse`): ein Newton-Verfahren, das in einem Intervall eingeschlossen ist und bei Bedarf auf Intervallhalbierung zurückfällt. Der Definitionsbereich ist $0 \le \operatorname{inv}\alpha \le \operatorname{inv}(89°)$; Werte außerhalb lösen einen typisierten Fehler aus, es wird nie ein Ersatzwert zurückgegeben.

**Genauigkeit.** Die Iteration endet, wenn das Residuum die Rundungsgrenze von $\tan\alpha$ erreicht. Für übliche Eingriffswinkel liegt der Fehler der Umkehrung bei etwa $10^{-15}$ rad. Für sehr kleine Winkel löscht sich $\tan\alpha - \alpha$ weitgehend aus; der Fehler der Umkehrung wächst dort mit $1/\alpha$ und ist im Test mit seiner Schranke festgehalten.

In [ ]:
alpha_n = math.radians(inputs["normal_pressure_angle_deg"]["value"])
beta = math.radians(inputs["helix_angle_deg"]["value"])
alpha_t = iv.transverse_pressure_angle(alpha_n, beta)

rows = []
for key, angle in (("inv_alpha_n_rad", alpha_n), ("inv_alpha_t_rad", alpha_t)):
    entry = expected[key]
    value = iv.inv(angle)
    assert abs(value - entry["value"]) <= printed_tolerance(entry["decimals"]) + 1e-9
    round_trip = iv.inv_inverse(value)
    assert abs(round_trip - angle) <= 1e-14
    rows.append(
        {
            "Benennung der Norm": entry["description"],
            "Zeichen": entry["symbol"],
            "gearcore": value,
            "Norm (gedruckt)": entry["printed"],
            "Winkel aus Umkehrung in Grad": math.degrees(round_trip),
            "Fehler der Umkehrung in rad": round_trip - angle,
        }
    )
pd.DataFrame(rows)

In [ ]:
for bad in (-1e-6, 1e3, float("nan")):
    try:
        iv.inv_inverse(bad)
    except gearcore.GearCoreError as error:
        print(f"{bad!r:>8}: {type(error).__name__}: {error}")

In [ ]:
def _plot_involute_function(marked_deg):
    angles = [0.5 * step for step in range(0, 91)]
    values = [iv.inv(math.radians(angle)) for angle in angles]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(angles, values, color="tab:blue")
    for angle in marked_deg:
        value = iv.inv(math.radians(angle))
        ax.plot([angle], [value], "o", color="tab:red")
        ax.annotate(
            f"{angle:g}°: {value:.5f}", (angle, value), textcoords="offset points", xytext=(6, -12)
        )
    ax.set_xlabel("Winkel α in Grad")
    ax.set_ylabel("inv α in rad")
    ax.set_title("Evolventenfunktion, DIN ISO 21771:2014-08 Gl. (18)")
    ax.grid(True, alpha=0.3)
    return fig


_plot_involute_function([14.5, 20.0, 25.0]);

## 3 Zahndicke und Lückenweite

**Normbezug.** DIN ISO 21771:2014-08: §4.3.3 Gl. (8), S. 28; §4.7.1 Gl. (38), (39), S. 35; §4.7.2 Gl. (40) bis (42), S. 35; §4.7.3 Gl. (43), S. 35, Gl. (44), S. 36; §4.7.4 Gl. (45) bis (47), S. 36; §4.7.5 Gl. (48), (49), S. 37; §4.7.6 Gl. (50), (51), S. 37.

**Formeln (eigene Transkription, Außenverzahnung).**

Zahndicke:

$$\psi = \frac{\pi + 4\,x\tan\alpha_{\mathrm{n}}}{2\,z} \quad (41) \qquad s_{\mathrm{t}} = m_{\mathrm{t}}\left(\frac{\pi}{2} + 2\,x\tan\alpha_{\mathrm{n}}\right) \quad (39)$$

$$\psi_{\mathrm{y}} = \psi + \operatorname{inv}\alpha_{\mathrm{t}} - \operatorname{inv}\alpha_{\mathrm{yt}} \quad (40) \qquad s_{\mathrm{yt}} = d_{\mathrm{y}}\,\psi_{\mathrm{y}} \quad (38) \qquad \psi_{\mathrm{b}} = \psi + \operatorname{inv}\alpha_{\mathrm{t}} \quad (42)$$

Lückenweite:

$$\eta = \frac{\pi - 4\,x\tan\alpha_{\mathrm{n}}}{2\,z} \quad (46) \qquad e_{\mathrm{t}} = m_{\mathrm{t}}\left(\frac{\pi}{2} - 2\,x\tan\alpha_{\mathrm{n}}\right) \quad (44)$$

$$\eta_{\mathrm{y}} = \eta - \operatorname{inv}\alpha_{\mathrm{t}} + \operatorname{inv}\alpha_{\mathrm{yt}} \quad (45) \qquad e_{\mathrm{yt}} = d_{\mathrm{y}}\,\eta_{\mathrm{y}} \quad (43) \qquad \eta_{\mathrm{b}} = \eta - \operatorname{inv}\alpha_{\mathrm{t}} \quad (47)$$

Normalschnitt:

$$s_{\mathrm{n}} = m_{\mathrm{n}}\left(\frac{\pi}{2} + 2\,x\tan\alpha_{\mathrm{n}}\right) \quad (49) \qquad e_{\mathrm{n}} = m_{\mathrm{n}}\left(\frac{\pi}{2} - 2\,x\tan\alpha_{\mathrm{n}}\right) \quad (51)$$

$$s_{\mathrm{yn}} = s_{\mathrm{yt}}\cos\beta_{\mathrm{y}} \quad (48) \qquad e_{\mathrm{yn}} = e_{\mathrm{yt}}\cos\beta_{\mathrm{y}} \quad (50) \qquad \tan\beta_{\mathrm{y}} = \tan\beta\,\frac{d_{\mathrm{y}}}{d} \quad (8)$$

Die Größen gelten für die Nennverzahnung mit dem Profilverschiebungsfaktor $x$. Die vom Werkzeug erzeugte Zahndicke (Erzeugungs-Profilverschiebung $x_{\mathrm{E}}$, Abmaße) folgt in Notebook 03 und 05.

**Rechenwert der Zahndicke am Teilzylinder.** Gl. (49) rechnet die beiden Evolventenflanken eines Zahns bis zum Teilzylinder weiter, auch wenn sie sich vorher schneiden. Bei stark negativer Profilverschiebung und großem Eingriffswinkel liegt der Schnittpunkt der Flanken (spitzer Zahn) unterhalb des Teilkreises; am Teilkreis gibt es dann keinen Zahn mehr und der Rechenwert wird negativ. Der wirkliche Zahn liegt weiter innen und hat dort eine positive Dicke. Das tritt erst für $|x| > \pi / (4\tan\alpha_{\mathrm{n}})$ auf: bei $\alpha_{\mathrm{n}} = 20°$ ab $|x| = 2{,}158$, also außerhalb des zugelassenen Bereichs $|x| \le 2$; innerhalb des Bereichs erst ab $\alpha_{\mathrm{n}} = 21{,}4°$.

**Vorzeichen.** Zahndicke, Lückenweite und ihre Halbwinkel am Teilzylinder sowie die Lücken-Halbwinkel $\eta_{\mathrm{y}}$ und $\eta_{\mathrm{b}}$ werden mit Vorzeichen zurückgegeben. Ein Wert kleiner oder gleich null bedeutet, dass auf diesem Zylinder kein Zahn oder keine Lücke vorhanden ist. Das kommt bei herstellbaren Rädern vor: $\eta_{\mathrm{b}}$ ist negativ, sobald der Fußkreis über dem Grundkreis liegt. Für den Teilzylinder meldet das Ergebnis den Fall als Hinweis. Anders Gl. (40): Ein negativer Zahndicken-Halbwinkel bei $d_{\mathrm{y}}$ heißt, dass $d_{\mathrm{y}}$ jenseits des spitzen Zahns liegt; das ist ein Fehler (`GeometryInfeasibleError`). Die Invarianten $s_{\mathrm{t}} + e_{\mathrm{t}} = \pi\,m_{\mathrm{t}}$ und $\psi + \eta = \pi / z$ sind als Eigenschaftstests im Paket abgesichert.

In [ ]:
pinion = gears["pinion"]
pd.DataFrame(
    [
        {
            "Programmname": name,
            "Zeichen": info.json_schema_extra["symbol"],
            "Benennung der Norm": info.json_schema_extra["designation"],
            "Wert": getattr(pinion, name),
            "Einheit": info.json_schema_extra["unit"],
            "Gleichung": info.json_schema_extra.get("equation", "Eingabe"),
        }
        for name, info in type(pinion).model_fields.items()
        if info.json_schema_extra is not None
    ]
)

In [ ]:
# Ergebnisse überstehen den Weg über JSON ohne Verlust einer Stelle (Projektregel 8a)
for gear in gears.values():
    restored = type(gear).model_validate_json(gear.model_dump_json())
    assert restored == gear and gear.warnings == ()

# Rechenwert am Teilzylinder: z = 100, m_n = 1, alpha_n = 30°, x = -1,5. Die Flanken schneiden sich
# bei d = 99,72 mm, also unterhalb des Teilkreises (100 mm); der Kopfkreis liegt bei 99 mm, dort ist
# der Zahn 0,40 mm dick. Gl. (49) liefert am Teilzylinder s_n < 0; das Ergebnis meldet es.
extreme = iv.compute_basic_gear_geometry(
    number_of_teeth=100,
    normal_module_mm=1.0,
    normal_pressure_angle_deg=30.0,
    helix_angle_deg=0.0,
    profile_shift_coefficient=-1.5,
)
assert extreme.normal_tooth_thickness_mm < 0.0
assert [warning.code for warning in extreme.warnings] == ["no_tooth_at_reference_cylinder"]
print("s_n =", extreme.normal_tooth_thickness_mm, "mm")
print(extreme.warnings[0].message)

In [ ]:
def _plot_tooth_thickness(gear, steps=200):
    alpha_t = math.radians(gear.transverse_pressure_angle_deg)
    psi = math.radians(gear.tooth_thickness_half_angle_deg)
    d_b, d = gear.base_diameter_mm, gear.reference_diameter_mm
    diameters, thickness = [], []
    for step in range(steps + 1):
        d_y = d_b + (1.3 * d - d_b) * step / steps
        alpha_yt = iv.transverse_profile_angle_at(d_y, d_b)
        try:
            psi_y = iv.tooth_thickness_half_angle_at(psi, alpha_t, alpha_yt)
        except gearcore.GeometryInfeasibleError:
            break  # jenseits des spitzen Zahns gibt es keine Zahndicke
        diameters.append(d_y)
        thickness.append(iv.transverse_tooth_thickness_at(d_y, psi_y))
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(diameters, thickness, color="tab:blue")
    ax.axvline(d_b, color="gray", linestyle=":", label=f"Grundkreis ${latex('d_b')}$")
    ax.axvline(d, color="gray", linestyle="--", label=f"Teilkreis ${latex('d')}$")
    ax.plot(
        [d],
        [gear.transverse_tooth_thickness_mm],
        "o",
        color="tab:red",
        label=f"${latex('s_t')}$ nach Gl. (39)",
    )
    ax.set_xlabel(f"Durchmesser ${latex('d_y')}$ in mm")
    ax.set_ylabel(f"Zahndicke ${latex('s_yt')}$ in mm")
    ax.set_title(
        f"Zahndicke im Stirnschnitt, z = {gear.number_of_teeth}, x = {gear.profile_shift_coefficient:g}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    return fig


_plot_tooth_thickness(pinion);

## 4 Bezugsprofil der Verzahnung

**Normbezug.** DIN 867:1986-02: §4.1 Gl. (2), S. 1; §4.3 Gl. (4), (5), S. 2; §4.4 (Kopfspiel); §4.5 Gl. (7), (8), S. 2; §4.6 Gl. (9), S. 3. ISO 53:1998: §5.1 Tabelle 2, S. 3; §5.9 Gl. (2), (3), S. 4; Anhang A, Tabelle A.1, S. 5.

**Festlegungen.** Profilwinkel $\alpha_{\mathrm{P}} = 20°$, Kopfhöhe $h_{\mathrm{aP}} = m$, Fußhöhe $h_{\mathrm{fP}} = m + c_{\mathrm{P}}$, Teilung $p = \pi\,m$. DIN 867 nennt für das Kopfspiel $c_{\mathrm{P}}$ den Bereich 0,1·$m$ bis 0,4·$m$ als üblich, nicht als Grenze. Ein Kopfspiel außerhalb davon wird angenommen und als Hinweis gemeldet.

**Formeln (eigene Transkription).** Der Fußrundungsradius ist doppelt begrenzt, der kleinere Wert gilt:

$$\rho_{\mathrm{fP}} \le \frac{c_{\mathrm{P}}}{1 - \sin\alpha_{\mathrm{P}}} \qquad \rho_{\mathrm{fP}} \le \frac{\pi\,m/4 - h_{\mathrm{fP}}\tan\alpha_{\mathrm{P}}}{\tan\left(\dfrac{90° - \alpha_{\mathrm{P}}}{2}\right)}$$

Die erste Grenze hält die Rundung unterhalb der gemeinsamen Zahnhöhe (DIN 867 Gl. (7), ISO 53 Gl. (2)), die zweite ist die Vollrundung der Lücke (ISO 53 Gl. (3), gleichwertig zu DIN 867 Gl. (8) mit $h_{\mathrm{fP}} = m + c_{\mathrm{P}}$). Bei $\alpha_{\mathrm{P}} = 20°$ wechselt die maßgebende Grenze bei $c_{\mathrm{P}} = 0{,}295\,m$. ISO 53 gibt Gl. (3) für $0{,}295\,m < c_{\mathrm{P}} \le 0{,}396\,m$ an; DIN 867 Gl. (8) hat keine obere Grenze und deckt damit auch Typ D ($c_{\mathrm{P}} = 0{,}4\,m$) ab. Die Fußformhöhe, bis zu der die Flanke gerade ist, beträgt

$$h_{\mathrm{FfP}} = h_{\mathrm{fP}} - \rho_{\mathrm{fP}}\,(1 - \sin\alpha_{\mathrm{P}}) \quad (9).$$

**Hinweis zur Rundung.** Die Normen drucken den Fußrundungsradius mit zwei Nachkommastellen. Zwei gedruckte Wertepaare überschreiten dadurch die Grenze: $c_{\mathrm{P}} = 0{,}25\,m$ mit $\rho_{\mathrm{fP}} = 0{,}38\,m$ (Grenze 0,37995·$m$; ISO 53 Typ A, DIN 867) und $c_{\mathrm{P}} = 0{,}3\,m$ mit $\rho_{\mathrm{fP}} = 0{,}45\,m$ (Grenze 0,44592·$m$; DIN 867, Erläuterungen). `gearcore` prüft die Grenze ohne Toleranz und lässt genau diese beiden gedruckten Paare zu; sie werden als Hinweis gemeldet. Jede andere Überschreitung löst `GeometryInfeasibleError` aus. Ein weiterer Tabellenwert von DIN 867 weicht von der Formel ab: Für $c_{\mathrm{P}} = 0{,}17\,m$ ist 0,25·$m$ gedruckt, Gl. (7) ergibt 0,2584·$m$.

In [ ]:
racks = [rack.iso53_basic_rack(kind) for kind in ("A", "B", "C", "D")]
rows = []
for profile in racks:
    alpha_p = math.radians(profile.profile_angle_deg)
    limit = rack.max_fillet_radius_factor(
        bottom_clearance_factor=profile.bottom_clearance_factor,
        dedendum_factor=profile.dedendum_factor,
        profile_angle_rad=alpha_p,
    )
    rack.validate_basic_rack(profile)
    rows.append(
        {
            "Bezugsprofil": profile.name,
            "h_aP*": profile.addendum_factor,
            "h_fP*": profile.dedendum_factor,
            "c_P*": profile.bottom_clearance_factor,
            "rho_fP*": profile.fillet_radius_factor,
            "rho_fP* max": limit,
            "Hinweise": ", ".join(f.code for f in rack.check_basic_rack(profile)) or "–",
            "h_FfP*": rack.root_form_height_factor(
                dedendum_factor=profile.dedendum_factor,
                fillet_radius_factor=profile.fillet_radius_factor,
                profile_angle_rad=alpha_p,
            ),
        }
    )
pd.DataFrame(rows)

In [ ]:
def _plot_basic_racks(profiles, points=40):
    fig, ax = plt.subplots(figsize=(8, 4))
    for profile in profiles:
        alpha = math.radians(profile.profile_angle_deg)
        rho, h_f, h_a = (
            profile.fillet_radius_factor,
            profile.dedendum_factor,
            profile.addendum_factor,
        )
        h_form = rack.root_form_height_factor(
            dedendum_factor=h_f, fillet_radius_factor=rho, profile_angle_rad=alpha
        )
        # rechte Flanke eines Zahns, Bezugslinie y = 0, Zahnmitte x = 0, alles je Modul
        xs = [math.pi / 4 - h_a * math.tan(alpha), math.pi / 4 + h_form * math.tan(alpha)]
        ys = [h_a, -h_form]
        centre_x = xs[-1] + rho * math.cos(alpha)
        centre_y = -h_f + rho
        for step in range(points + 1):
            angle = math.pi + alpha + (math.pi / 2 - alpha) * step / points
            xs.append(centre_x + rho * math.cos(angle))
            ys.append(centre_y + rho * math.sin(angle))
        xs.append(math.pi / 2)
        ys.append(ys[-1])
        full_x = [-x for x in reversed(xs)] + xs
        full_y = list(reversed(ys)) + ys
        ax.plot(full_x, full_y, label=f"{profile.name}: ${latex('rho_fP*')}$ = {rho:g}")
    ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set_xlabel("x / m")
    ax.set_ylabel("y / m")
    ax.set_title("Bezugsprofile nach ISO 53:1998, Tabelle A.1 (ein Zahn, je Modul)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    return fig


_plot_basic_racks(racks);

## 5 Werkzeug-Bezugsprofil

**Normbezug.** DIN ISO 21771:2014-08, §7.1, Bild 35, S. 63 (Bezeichnungen des Werkzeug-Bezugsprofils); DIN 867:1986-02, §4.5, Anmerkung, S. 2; DIN 3972:1952-02, Tabelle S. 1 und Erläuterungen S. 2.

**Zusammenhang.** Das Werkzeug-Bezugsprofil ist das Gegenprofil des Bezugsprofils der Verzahnung: Der Werkzeugkopf erzeugt den Zahnfuß. Damit gilt $h_{\mathrm{aP0}} = h_{\mathrm{fP}}$ und $\rho_{\mathrm{aP0}} = \rho_{\mathrm{fP}}$.

**Formelzeichen der Kopfrundung.** `gearcore` verwendet $\rho_{\mathrm{aP0}}$. So schreiben es die Gleichungen der neuesten Geometrienorm (DIN ISO 21771:2014-08, Gl. (128) im normativen Anhang NB, S. 6, und Gl. (130), S. 69), ISO/TR 6336-30:2022 (Table 2, S. 5) und DIN 867:1986-02 (§4.4). ISO 6336-3:2019 (Table 2, S. 7) und VDI 2736 Blatt 2 (S. 21) schreiben $\rho_{\mathrm{a0}}$; die Beschriftung von Bild 35 in DIN ISO 21771 zeigt $r_{\mathrm{aP0}}$.

**Schreibweise.** DIN 3972:1952-02 verwendet ältere Formelzeichen, STplus schreibt griechische Buchstaben aus (`alfa_t`, `ro_aP0`). `gearcore` rechnet in der Schreibweise der neuesten gültigen Norm; die Zuordnung steht je Größe in der Registry `data/quantities.yaml` und wird unten ausgegeben (Zeichen der aktuellen Norm, Zeichen in STplus, Zeichen der ersetzten Normen).

**Formeln in heutiger Schreibweise (eigene Transkription).**

| Bezugsprofil | Zweck | Kopfhöhe des Werkzeugs |
|---|---|---|
| I | Fertigbearbeitung | $h_{\mathrm{aP0}} = 1{,}167\,m$ |
| II | Fertigbearbeitung | $h_{\mathrm{aP0}} = 1{,}25\,m$ |
| III | Vorbearbeitung zum Schleifen oder Schaben, bis Modul 8 | $h_{\mathrm{aP0}} = 1{,}25\,m + 0{,}25\,\sqrt[3]{m}$ |
| IV | Vorbearbeitung zum Schlichten | $h_{\mathrm{aP0}} = 1{,}25\,m + 0{,}6\,\sqrt[3]{m}$ |

Für die Bezugsprofile III und IV bleibt eine Bearbeitungszugabe je Flanke stehen, senkrecht zur Flanke gemessen: $q = (h_{\mathrm{aP0}} - 1{,}25\,m)\,\sin\alpha_{\mathrm{P0}}$. Die Bezugsprofile I und II sind Fertigbearbeitungsprofile mit $q = 0$. Die Formeln für III und IV sind Zahlenwertgleichungen mit $m$ in mm. Rechenbeispiel der Norm (S. 2): Bezugsprofil III, Modul 8 ergibt 0,17 mm.

**Kopfrundung.** Die Rundung am Werkzeugkopf ist in der Norm je Modul tabelliert (0,08 mm bei Modul 1 bis 3,20 mm bei Modul 16). Der dort genannte Wert 0,2·$m$ ist ein Näherungswert und wird nicht als Formel verwendet. Für Moduln außerhalb der Tabelle gibt es keine Voreinstellung.

**Befund.** Von 152 Tabellenwerten der Norm entsprechen 146 dem gerundeten Formelwert. Sechs Werte weichen um mehr als eine halbe Einheit der letzten Stelle ab, am stärksten Modul 4,5, Bezugsprofil IV: gedruckt 6,60 mm, Formel 6,6156 mm. `gearcore` rechnet mit den Formeln; die Abweichungen sind im Test einzeln festgehalten.

In [ ]:
# Größen dieses Notebooks aus der Registry: aktuelle Norm, STplus, ersetzte Normen
shown = [
    "tool_profile_angle",
    "tool_addendum",
    "tool_tip_radius",
    "tool_dedendum",
    "machining_allowance",
    "pitch",
    "basic_rack_dedendum",
    "normal_pressure_angle",
    "transverse_pressure_angle",
    "base_helix_angle",
    "centre_distance",
]
registry = quantities()
pd.DataFrame(
    [
        {
            "Programmname": name,
            "Zeichen": registry[name].symbol,
            "Benennung der Norm": registry[name].description,
            "Quelle": registry[name].source,
            "STplus": registry[name].stplus.symbol if registry[name].stplus else "–",
            "ersetzte Zeichen": ", ".join(
                f"{old.symbol} ({old.source})" for old in registry[name].replaced
            )
            or "–",
        }
        for name in shown
    ]
)

In [ ]:
module_mm = 4.5
rows = []
for profile in ("I", "II", "III", "IV"):
    tool = rack.din3972_tool(profile, module_mm)
    rows.append(
        {
            "Bezugsprofil": profile,
            "h_aP0 in mm": rack.din3972_tool_addendum_mm(profile, module_mm),
            "h_aP0*": tool.addendum_factor,
            "rho_aP0*": tool.tip_radius_factor,
            "q in mm": tool.machining_allowance_mm,
        }
    )
counterpart = rack.tool_from_basic_rack(rack.iso53_basic_rack("A"))
assert counterpart.addendum_factor == 1.25 and counterpart.tip_radius_factor == 0.38
pd.DataFrame(rows)

## 6 Modulreihe

**Normbezug.** ISO 54:1996, §3, Tabelle 1, S. 1 und 2.

Die Norm nennt die Moduln 1 mm bis 50 mm in zwei Reihen; Reihe I ist zu bevorzugen. Ein Modul außerhalb der Reihen ist zulässig und wird als Hinweis gemeldet, nicht abgelehnt. Das betrifft kleine Moduln der Kunststoffverzahnungen.

In [ ]:
rows = []
for module_mm in (2.0, 4.5, 6.5, 0.5):
    finding = rack.check_module(module_mm)
    rows.append(
        {
            "Modul in mm": module_mm,
            "Reihe": rack.module_series(module_mm) or "–",
            "Hinweis": finding.message if finding is not None else "–",
        }
    )
assert rack.check_module(2.0) is None and rack.check_module(0.5) is not None
pd.DataFrame(rows)

## 7 Vergleich mit STplus

STplus 11.1F dient als Zahlenorakel (ADR-102). Verglichen wird Wert für Wert gegen das Listing und, bei eigenen Läufen, gegen die Schnittstellendatei. Das Listing druckt Längen mit drei, Winkel und den Stirnmodul mit fünf und den Profilverschiebungsfaktor mit vier Nachkommastellen; die Schnittstellendatei druckt fünf Nachkommastellen.

**Rechengenauigkeit.** STplus rechnet mit einfacher Genauigkeit (32 Bit, etwa sieben signifikante Stellen), `gearcore` mit doppelter Genauigkeit (64 Bit, etwa sechzehn Stellen). Nachweis an den fünf Grundgrößen der elf eigenen Läufe (99 gedruckte Werte): Dieselben Formeln, in 32 Bit ausgewertet, treffen alle 99 Werte; das 64-Bit-Ergebnis trifft 93. Die sechs Abweichungen liegen in den beiden schrägverzahnten Läufen; bei Geradverzahnung sind die Grundgrößen unempfindlich. Die Abweichung ist damit eine Eigenschaft des Zahlenformats von STplus und kein Unterschied in den Formeln (ADR-106, `tests/test_stplus_parity.py`).

**Toleranz.** Je Wert werden drei Anteile getrennt ausgewiesen: die halbe Einheit der letzten gedruckten Stelle, zwei Schritte der 32-Bit-Zahlen am gedruckten Wert und, wo STplus den Profilverschiebungsfaktor selbst bestimmt hat, die Rundung des gedruckten $x$. Der Vergleich erkennt damit relative Fehler ab etwa 5·10⁻⁷.

**Profilverschiebung.** STplus übernimmt $x_1$ aus der Eingabe. Ist ein Achsabstand gegeben, bestimmt STplus $x_2$ aus dem Achsabstand und überschreibt ein eingegebenes $x_2$ (kst-E: Eingabe 0,3143, verwendet 0,31433). Ist eine Zahnweite gegeben, folgt $x$ aus der Zahnweite.

**Urteil je Wert.**

| Urteil | Bedeutung |
|---|---|
| `identical` | gleich auf die von STplus gedruckte Stellenzahl |
| `oracle_accuracy` | Abweichung durch die Genauigkeit des STplus-Werts erklärt: einfache Genauigkeit oder gerundet gedruckter Profilverschiebungsfaktor |
| `different` | nicht erklärt: Fehler oder dokumentierter Normunterschied |

Referenzpaare mit Vertrauensstufe `unverified` (kst-A, kst-C) werden mitgeführt, zählen aber nicht als harte Referenz.

In [ ]:
parity = pd.DataFrame(
    [
        row.model_dump(mode="json")
        for case in stplus_case_dirs()
        for row in compare_basic_geometry(case.name)
    ]
)
summary = parity.groupby(["origin", "verdict"]).size().unstack(fill_value=0)
hard = parity[parity["trust"] != "unverified"]
assert (hard["verdict"] != ParityVerdict.DIFFERENT.value).all()
print(f"{parity['case'].nunique()} Fälle, {len(parity)} Vergleiche")
summary

In [ ]:
columns = [
    "case",
    "trust",
    "origin",
    "gear",
    "symbol",
    "gearcore_value",
    "stplus_token",
    "difference",
    "verdict",
]
parity.loc[parity["verdict"] != ParityVerdict.IDENTICAL.value, columns].reset_index(drop=True)

In [ ]:
def _plot_parity(frame):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
    colours = {"identical": "tab:green", "oracle_accuracy": "tab:orange", "different": "tab:red"}
    for ax, origin in zip(axes, ("listing", "interface"), strict=True):
        part = frame[frame["origin"] == origin].reset_index(drop=True)
        units = part["difference"] / (2.0 * part["print_tolerance"])
        for verdict, colour in colours.items():
            mask = part["verdict"] == verdict
            ax.scatter(part.index[mask], units[mask], s=12, color=colour, label=verdict)
        ax.axhspan(-0.5, 0.5, color="tab:green", alpha=0.08)
        ax.set_title(f"{origin}: {len(part)} Werte")
        ax.set_xlabel("Vergleich (laufende Nummer)")
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel("gearcore − STplus in Einheiten\nder letzten gedruckten Stelle")
    axes[1].legend(fontsize=8)
    fig.suptitle("Abweichung gearcore (64 Bit) gegen STplus 11.1F (32 Bit)")
    fig.tight_layout()
    return fig


_plot_parity(parity);

## 8 Rückverfolgbarkeit

Jede Rechenfunktion trägt ihren Normbezug als `@eq`-Angabe mit Quelle und Ausgabe, Gleichung, Abschnitt und Seite (ADR-103). Die Tabelle wird aus dem Paket gelesen, nicht von Hand gepflegt.

In [ ]:
rows = []
for module in (iv, rack):
    for name, function in inspect.getmembers(module, inspect.isfunction):
        if name.startswith("_") or function.__module__ != module.__name__:
            continue
        references = equations_of(function)
        if name in getattr(module, "EQ_EXEMPT", ()):
            continue  # Orchestratoren und Prüfungen rufen nur getragene Funktionen auf
        assert references, name
        rows.append(
            {
                "Funktion": f"{module.__name__}.{name}",
                "Normbezug": "; ".join(str(reference) for reference in references),
            }
        )
pd.DataFrame(rows)